# stoX — TFT Training on Colab GPU

**Before running anything:**
1. Go to **Runtime → Change runtime type → T4 GPU** (free tier) or A100 (Colab Pro)
2. Upload `sl20_feature_panel.parquet` to your Google Drive (any folder is fine)
3. Run cells top to bottom — each section has a ✅ checkpoint

When training finishes, the model checkpoint is saved to your Drive automatically.
You then download `best.ckpt` and drop it into `ml/models/tft_v1/` on your local machine.

## Step 1 — Verify GPU

In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU")

## Step 2 — Clone the stoX repo

In [ ]:
import os

REPO_DIR = "/content/stoX"

if os.path.exists(REPO_DIR):
    # Pull latest if already cloned
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Dineth-San/stoX.git {REPO_DIR}

print("\n✅ Repo ready at", REPO_DIR)

## Step 3 — Mount Google Drive and copy the parquet file

After mounting, update `PARQUET_IN_DRIVE` to the exact path where you uploaded 
`sl20_feature_panel.parquet` in your Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── UPDATE THIS PATH to where you put the file in Drive ──────────────────────
PARQUET_IN_DRIVE = "/content/drive/MyDrive/sl20_feature_panel.parquet"
# ─────────────────────────────────────────────────────────────────────────────

DEST = f"{REPO_DIR}/ml/data/features/sl20_feature_panel.parquet"
os.makedirs(os.path.dirname(DEST), exist_ok=True)

import shutil
shutil.copy(PARQUET_IN_DRIVE, DEST)

size_mb = os.path.getsize(DEST) / 1e6
print(f"✅ Parquet copied: {size_mb:.1f} MB → {DEST}")

## Step 4 — Install dependencies

In [ ]:
# Install the key ML deps. Colab already has torch/numpy/pandas.
# pytorch-forecasting pulls in lightning automatically.
!pip install -q pytorch-forecasting>=1.0 lightning>=2.0 mlflow>=2.14 pandera>=0.20

# Install the sl20_ml package in editable mode
!pip install -q -e {REPO_DIR}/ml

print("✅ Dependencies installed")

## Step 5 — Set up output directory in Google Drive

The trained model checkpoint will be saved here so it persists after the Colab session ends.

In [ ]:
# Where the checkpoint will be saved on Drive
DRIVE_MODEL_DIR = "/content/drive/MyDrive/stoX_models/tft_v1"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# Symlink the model dir inside the repo → Drive
# (so train_model.py saves directly to Drive)
LOCAL_MODEL_DIR = f"{REPO_DIR}/ml/models/tft_v1"
if os.path.islink(LOCAL_MODEL_DIR):
    os.unlink(LOCAL_MODEL_DIR)
elif os.path.isdir(LOCAL_MODEL_DIR):
    shutil.rmtree(LOCAL_MODEL_DIR)

os.makedirs(f"{REPO_DIR}/ml/models", exist_ok=True)
os.symlink(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)

print(f"✅ Model output → {DRIVE_MODEL_DIR}")
print("   (checkpoint saves directly to your Google Drive)")

## Step 6 — Train the model

Expected time on a T4 GPU: **~8–15 minutes** for 30–50 epochs (early stopping kicks in).

Watch the `val_loss` column — it should decrease epoch by epoch. When it stops improving for 15 epochs, training stops automatically.

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/ml/src")
os.chdir(f"{REPO_DIR}/ml")

# Run the training script
%run {REPO_DIR}/ml/train_model.py

## Step 7 — Verify the checkpoint was saved

In [ ]:
import json
from pathlib import Path

model_dir = Path(DRIVE_MODEL_DIR)
ckpt_files = list(model_dir.glob("*.ckpt"))
config_file = model_dir / "model_config.json"

print("Files in model directory:")
for f in model_dir.iterdir():
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

if config_file.exists():
    cfg = json.loads(config_file.read_text())
    print("\n✅ Training complete! Results:")
    print(f"  Val  → MAE={cfg['val_metrics']['mae']:.4f}  "
          f"DA={cfg['val_metrics']['directional_accuracy']:.1%}  "
          f"QC={cfg['val_metrics']['quantile_coverage']:.1%}")
    print(f"  Test → MAE={cfg['test_metrics']['mae']:.4f}  "
          f"DA={cfg['test_metrics']['directional_accuracy']:.1%}  "
          f"QC={cfg['test_metrics']['quantile_coverage']:.1%}")
else:
    print("⚠️  model_config.json not found — training may have failed")

## Step 8 — Download the checkpoint to your local machine

Two options:

**Option A (recommended):** The checkpoint is already in your Google Drive at `stoX_models/tft_v1/`. Just download it from drive.google.com.

**Option B:** Download directly from this notebook:

In [ ]:
from google.colab import files

# Download checkpoint
ckpt = next(model_dir.glob("*.ckpt"), None)
if ckpt:
    files.download(str(ckpt))
    print(f"Downloading: {ckpt.name}")
else:
    print("No checkpoint found.")

# Download model_config.json
if config_file.exists():
    files.download(str(config_file))
    print("Downloading: model_config.json")

## Step 9 — What to do with the downloaded files

Once you have `best.ckpt` and `model_config.json` on your local machine:

```
# Place them here:
D:\stox\ml\models\tft_v1\best.ckpt
D:\stox\ml\models\tft_v1\model_config.json
```

Then test the model locally:
```powershell
cd D:\stox\ml
python predict.py
```

That's it — your locally-served prediction script will use the GPU-trained checkpoint.

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `batch_size` in `ml/configs/pipeline.yaml` from 64 → 32 |
| `ModuleNotFoundError: pytorch_forecasting` | Re-run Step 4 |
| `FileNotFoundError: sl20_feature_panel.parquet` | Re-run Step 3 and check the Drive path |
| Session disconnected mid-training | Checkpoint saves after each epoch — it won't restart from scratch. Re-run Step 6 and it will continue |
| Drive not mounted | Re-run Step 3 |

**Runtime tip:** Keep the Colab tab active (don't let it idle). Free Colab disconnects after ~90 minutes of inactivity.